# Movie Recommender

In [8]:
import os

import numpy as np
import pandas as pd

RANDOM_SEED = 42

## 1. Load and clean the data

In [9]:
DATASET_DIRECTORY = os.path.join('data', 'ml-32m')

ratings = pd.read_csv(os.path.join(DATASET_DIRECTORY, 'ratings.csv'))
movies = pd.read_csv(os.path.join(DATASET_DIRECTORY, 'movies.csv'))
print(f'Ratings shape: {ratings.shape}')
print(f'Movies shape: {movies.shape}')

Ratings shape: (32000204, 4)
Movies shape: (87585, 3)


In [10]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


In [11]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [12]:
MIN_RATINGS_PER_USER = 10

ratings = ratings.dropna(subset=['userId', 'movieId', 'rating', 'timestamp'])
ratings = ratings[ratings.rating.between(0.5, 5.0)]
ratings = ratings.drop_duplicates(['userId', 'movieId'])

# For every row, count how many ratings that row's user has,
# then keep only the rows that belong to users with enough ratings.
ratings_per_user = ratings.groupby('userId').movieId.transform('size')
ratings = ratings[ratings_per_user >= MIN_RATINGS_PER_USER].reset_index(drop=True)

## 2. Training split

In [13]:
from helpers.temporal_split import temporal_split

train, val, test = temporal_split(ratings)
print(len(train), len(val), len(test))

25605546 3197329 3197329


## 3. Scoring functions

- **RMSE / MAE:** how far off the predicted star ratings are (lower is better).
- **Precision@K / Recall@K / NDCG@K:** how good the top-K recommendations are (higher is better). A recommendation counts as correct if it is a held-out movie the user rated 4 stars or more.

In [14]:
from helpers.scoring import mae, rmse

def ndcg_at_k(recommended, relevant, k):
    credit = []
    for rank in range(k):
        credit.append(1.0 / np.log2(rank + 2))

    top_k = recommended[:k]
    score = 0.0
    for rank in range(len(top_k)):
        if top_k[rank] in relevant:
            score += credit[rank]

    # Best possible score: every liked movie sits at the very top of the list.
    hits_possible = min(len(relevant), k)
    best_possible = 0.0
    for rank in range(hits_possible):
        best_possible += credit[rank]

    if best_possible > 0:
        return score / best_possible
    return 0.0

def movies_by_user(df):
    result = {}
    for user_id, user_movies in df.groupby('userId').movieId:
        result[user_id] = set(user_movies.tolist())
    return result

def evaluate_ranking(recommend, history, held_out, cutoffs=(5, 10, 20), like_threshold=4.0):
    seen = movies_by_user(history)
    liked = held_out[held_out.rating >= like_threshold]
    relevant = movies_by_user(liked)

    scores = {}
    for k in cutoffs:
        scores[f'P@{k}'] = []
        scores[f'R@{k}'] = []
        scores[f'NDCG@{k}'] = []

    biggest_k = max(cutoffs)
    for user_id in relevant:
        correct = relevant[user_id]
        already_seen = seen.get(user_id, set())
        recommended = recommend(user_id, already_seen, biggest_k)

        for k in cutoffs:
            hits = 0
            for movie in recommended[:k]:
                if movie in correct:
                    hits += 1
            scores[f'P@{k}'].append(hits / k)
            scores[f'R@{k}'].append(hits / len(correct))
            scores[f'NDCG@{k}'].append(ndcg_at_k(recommended, correct, k))

    # Average each metric over all users.
    averages = {}
    for name in scores:
        averages[name] = float(np.mean(scores[name]))
    averages['n_users'] = len(relevant)
    return averages

## 4. Train movie embeddings

In [ ]:
from gensim.models import Word2Vec

LIKE_THRESHOLD = 3.5

train_by_time = train.sort_values(['userId', 'timestamp'])
liked = train_by_time[train_by_time.rating >= LIKE_THRESHOLD]

liked_movies = {}
for user_id, user_movies in liked.groupby('userId').movieId:
    liked_movies[user_id] = user_movies.tolist()

# Word2Vec expects "sentences" made of strings, so turn every movie ID into a string.
sequences = []
total_liked = 0
for user_id in liked_movies:
    sentence = []
    for movie in liked_movies[user_id]:
        sentence.append(str(movie))
    sequences.append(sentence)
    total_liked += len(sentence)

print(f'{len(sequences)} user sequences, avg length {total_liked / len(sequences):.1f}')

model = Word2Vec(
    sequences,
    vector_size=64,
    window=10,
    min_count=5,
    sg=1,
    negative=10,
    epochs=30,
    seed=RANDOM_SEED,
)
model.save("movie2vec.model")

200740 user sequences, avg length 81.2


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

In [16]:
model = Word2Vec.load("movie2vec.model")

Issue in: https://github.com/piskvorky/gensim/pull/3668, need to figure out how to handle for later, couldn't find a fix

In [17]:
# Print the closest embedded movies to a title that matches the search text.
movie_title = dict(zip(movies.movieId, movies.title))

def show_similar_movies(title_text, n=10):
    matches = movies[movies.title.str.contains(title_text, case=False, regex=False)]
    for movie_id in matches.movieId:
        if str(movie_id) in model.wv:
            print(f'Query: {movie_title[movie_id]}:')
            similar_movies = model.wv.most_similar(str(movie_id), topn=n)
            for similar_id, similarity in similar_movies:
                print(f'{similarity:.3f}  {movie_title[int(similar_id)]}')
            return
    print('No embedded movie found for', title_text)

show_similar_movies('Saw (2003)')

Query: Saw (2003):
0.807  Saw VII 3D - The Final Chapter (2010)
0.807  Saw V (2008)
0.797  Hush (2016)
0.795  Saw VI (2009)
0.794  Collector, The (2009)
0.779  Jigsaw (2017)
0.777  Final Destination, The (Final Destination 4) (Final Destination in 3-D, The) (2009)
0.775  Sinister 2 (2015)
0.768  Visit, The (2015)
0.764  The Conjuring 2 (2016)


## 5. Recommend movies with the embeddings

A user's **profile** is the average of the vectors of movies they liked in their training history. Recommend unseen movies whose vectors are closest to that profile. If a user has no usable profile, this notebook returns an empty recommendation list.

In [ ]:
vectors = model.wv.get_normed_vectors()

movie_ids = []
row_of = {}
for row in range(len(model.wv.index_to_key)):
    movie_id = int(model.wv.index_to_key[row])
    movie_ids.append(movie_id)
    row_of[movie_id] = row

liked_rows = {}
liked_row_sets = {}
vector_sums = {}

for user_id in liked_movies:
    rows = []
    for movie in liked_movies[user_id]:
        if movie in row_of:
            rows.append(row_of[movie])
    if len(rows) == 0:
        continue
    liked_rows[user_id] = rows
    liked_row_sets[user_id] = set(rows)
    vector_sums[user_id] = vectors[rows].sum(axis=0)

def user_profile(user_id, leave_out_row=None):
    if user_id not in liked_rows:
        return None

    total = vector_sums[user_id]
    count = len(liked_rows[user_id])

    if leave_out_row is not None and leave_out_row in liked_row_sets[user_id]:
        total = total - vectors[leave_out_row]
        count = count - 1

    if count <= 0:
        return None

    average = total / count
    length = np.linalg.norm(average)
    if length == 0:
        return None
    return average / length

def recommend_item2vec(user_id, seen, n):
    profile = user_profile(user_id)
    if profile is None:
        return []

    scores = np.dot(vectors, profile)

    best_first = np.argsort(scores)[::-1]

    recommendations = []
    for row in best_first:
        movie = int(movie_ids[row])
        if movie not in seen:
            recommendations.append(movie)
        if len(recommendations) == n:
            break
    return recommendations

print('item2vec', evaluate_ranking(recommend_item2vec, train, val, cutoffs=(5,10)))

item2vec {'P@5': 0.045213048560195514, 'R@5': 0.04938803063908386, 'NDCG@5': 0.05740962066378709, 'P@10': 0.040398469875677405, 'R@10': 0.08717533829410751, 'NDCG@10': 0.06921902638719145, 'n_users': 188220}


## 6. Predict star ratings with the embeddings

In [ ]:
from sklearn.linear_model import Ridge

def embedding_features(df):
    features = np.zeros((len(df), 1 + vectors.shape[1]), dtype=np.float32)

    user_list = df.userId.values
    movie_list = df.movieId.values
    for i in range(len(df)):
        row = row_of.get(movie_list[i])
        if row is None:
            continue
        profile = user_profile(user_list[i], leave_out_row=row)
        if profile is None:
            continue
        features[i, 0] = np.dot(vectors[row], profile)
        features[i, 1:] = vectors[row] * profile
    return features

ridge = Ridge(alpha=10.0).fit(embedding_features(train), train.rating.values)

def predict_item2vec_ratings(df):
    "Predict ratings and keep them within the dataset's 0.5-5 star range."
    return np.clip(ridge.predict(embedding_features(df)), 0.5, 5.0)

guesses = predict_item2vec_ratings(val)
print(f'item2vec val RMSE={rmse(val.rating, guesses):.4f} MAE={mae(val.rating, guesses):.4f}')

item2vec val RMSE=0.9730 MAE=0.7614


## 7. Evaluate the item2vec model

The final cell reports rating error on validation and test data, plus top-K recommendation quality on test data. Validation ratings are treated as already seen when evaluating test recommendations.

In [21]:
rating_rows = []
for split_name, split in [('Validation', val), ('Test', test)]:
    guesses = predict_item2vec_ratings(split)
    rating_rows.append({
        'Split': split_name,
        'RMSE': rmse(split.rating, guesses),
        'MAE': mae(split.rating, guesses),
    })
rating_results = pd.DataFrame(rating_rows).set_index('Split').round(4)
display(rating_results)

history = pd.concat([train, val])
ranking_results = evaluate_ranking(recommend_item2vec, history, test, cutoffs=(5,10))
users_evaluated = ranking_results.pop('n_users')
display(pd.DataFrame([ranking_results]).round(4))
print('Users with at least one liked test movie:', users_evaluated)

,RMSE,MAE
Split,,
Validation,0.9730,0.7614
Test,0.9895,0.7789


,P@5,R@5,NDCG@5,P@10,R@10,NDCG@10
0,0.0391,0.0396,0.0492,0.0344,0.0691,0.0575


Users with at least one liked test movie: 188901
